# Docker End-to-End Guide

This notebook covers Docker from basics to advanced topics:

1. **Basics** — Images, Containers, Lifecycle
2. **Dockerfile** — Building custom images
3. **Volumes & Networking** — Persistent data and container communication
4. **Docker Compose** — Multi-container applications
5. **Multi-stage Builds** — Optimized production images
6. **Best Practices** — Security, layer caching, .dockerignore

> **Prerequisites**: Docker Desktop installed and running.

---
## 1. Docker Basics

### 1.1 Verify Installation

In [ ]:
%%bash
docker --version
docker info --format '{{.ServerVersion}}'

### 1.2 Images — Pull, List, Inspect, Remove

In [ ]:
%%bash
# Pull an image from Docker Hub
docker pull alpine:3.19

# List local images
docker images

In [ ]:
%%bash
# Inspect image metadata (layers, env vars, entrypoint)
docker inspect alpine:3.19 --format '{{.Config.Cmd}} | Layers: {{len .RootFS.Layers}}'

In [ ]:
%%bash
# View layer history
docker history alpine:3.19

### 1.3 Containers — Run, List, Logs, Exec, Stop, Remove

In [ ]:
%%bash
# Run a container (foreground, auto-remove)
docker run --rm alpine:3.19 echo "Hello from inside a container!"

In [ ]:
%%bash
# Run a detached container with a name
docker run -d --name my-nginx -p 8080:80 nginx:alpine

# List running containers
docker ps

In [ ]:
%%bash
# View container logs
docker logs my-nginx --tail 5

In [ ]:
%%bash
# Execute a command inside a running container
docker exec my-nginx cat /etc/nginx/nginx.conf | head -10

In [ ]:
%%bash
# Resource usage stats
docker stats my-nginx --no-stream

In [ ]:
%%bash
# Stop and remove the container
docker stop my-nginx
docker rm my-nginx

### 1.4 Container Lifecycle Diagram

```
  Image
    │
    ▼  docker run
 Created ──► Running ──► Stopped ──► Removed
                │              ▲
                └── pause ──► Paused
```

---
## 2. Building Images with Dockerfile

### 2.1 A Simple Python App

In [ ]:
%%bash
mkdir -p /tmp/docker-demo/app

# Create a simple Flask app
cat > /tmp/docker-demo/app/main.py << 'PYEOF'
from flask import Flask, jsonify
import os, socket

app = Flask(__name__)

@app.route("/")
def home():
    return jsonify(
        message="Hello from Docker!",
        hostname=socket.gethostname(),
        env=os.getenv("APP_ENV", "development")
    )

@app.route("/health")
def health():
    return jsonify(status="healthy")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
PYEOF

cat > /tmp/docker-demo/app/requirements.txt << 'EOF'
flask==3.1.0
EOF

echo "App files created."

### 2.2 Write a Dockerfile

Key instructions: `FROM`, `WORKDIR`, `COPY`, `RUN`, `EXPOSE`, `CMD`

In [ ]:
%%bash
cat > /tmp/docker-demo/Dockerfile << 'DEOF'
# Use a slim Python base image
FROM python:3.12-slim

# Set working directory
WORKDIR /app

# Copy requirements first (layer caching optimization)
COPY app/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app/ .

# Set environment variable
ENV APP_ENV=production

# Expose port
EXPOSE 5000

# Health check
HEALTHCHECK --interval=30s --timeout=3s \
    CMD curl -f http://localhost:5000/health || exit 1

# Run the app
CMD ["python", "main.py"]
DEOF

echo "Dockerfile created."
cat /tmp/docker-demo/Dockerfile

### 2.3 Build & Run

In [ ]:
%%bash
# Build the image
docker build -t demo-flask:v1 /tmp/docker-demo/

# Check image size
docker images demo-flask:v1

In [ ]:
%%bash
# Run the container
docker run -d --name flask-app -p 5050:5000 demo-flask:v1

# Wait for startup and test
sleep 2
curl -s http://localhost:5050/ | python3 -m json.tool

In [ ]:
%%bash
# Cleanup
docker stop flask-app && docker rm flask-app

### 2.4 .dockerignore

In [ ]:
%%bash
cat > /tmp/docker-demo/.dockerignore << 'EOF'
__pycache__
*.pyc
.git
.env
*.md
.vscode
EOF

echo ".dockerignore created — keeps build context small and secure."

---
## 3. Volumes & Bind Mounts

### 3.1 Named Volumes (managed by Docker)

In [ ]:
%%bash
# Create a named volume
docker volume create demo-data

# Write data into the volume via a container
docker run --rm -v demo-data:/data alpine:3.19 sh -c 'echo "persisted!" > /data/test.txt'

# Read it from a different container — data persists!
docker run --rm -v demo-data:/data alpine:3.19 cat /data/test.txt

In [ ]:
%%bash
# Inspect volume metadata
docker volume inspect demo-data

# Cleanup
docker volume rm demo-data

### 3.2 Bind Mounts (map host directory into container)

Useful for development — code changes reflect immediately.

In [ ]:
%%bash
# Mount current host directory into the container
docker run --rm -v /tmp/docker-demo/app:/mounted alpine:3.19 ls /mounted

### 3.3 Volume Types Comparison

| Type | Syntax | Use Case |
|------|--------|----------|
| **Named Volume** | `-v mydata:/path` | Database storage, persistent state |
| **Bind Mount** | `-v /host/path:/path` | Dev hot-reload, config files |
| **tmpfs** | `--tmpfs /path` | Secrets, temp data (RAM only) |

---
## 4. Networking

### 4.1 Network Types

In [ ]:
%%bash
# List default networks
docker network ls

### 4.2 Custom Bridge Network (containers talk by name)

In [ ]:
%%bash
# Create a custom network
docker network create demo-net

# Start two containers on the same network
docker run -d --name server --network demo-net nginx:alpine
docker run --rm --network demo-net alpine:3.19 \
    sh -c 'apk add -q curl && curl -s http://server/ | head -5'

# The client resolved "server" via Docker DNS — no IPs needed!

In [ ]:
%%bash
# Cleanup
docker stop server && docker rm server
docker network rm demo-net

### 4.3 Network Types Comparison

| Driver | Isolation | DNS | Use Case |
|--------|-----------|-----|----------|
| **bridge** (default) | Yes | No | Single container |
| **bridge** (custom) | Yes | Yes | Multi-container apps |
| **host** | No | N/A | Max performance, no port mapping |
| **none** | Full | No | Security-sensitive workloads |

---
## 5. Docker Compose — Multi-Container Apps

A Flask API + Redis cache example.

In [ ]:
%%bash
mkdir -p /tmp/docker-compose-demo/app

# Flask app that uses Redis
cat > /tmp/docker-compose-demo/app/main.py << 'PYEOF'
from flask import Flask, jsonify
import redis, os

app = Flask(__name__)
cache = redis.Redis(host=os.getenv("REDIS_HOST", "redis"), port=6379, decode_responses=True)

@app.route("/")
def home():
    count = cache.incr("hits")
    return jsonify(message="Hello!", visit_count=count)

@app.route("/health")
def health():
    cache.ping()
    return jsonify(status="healthy")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
PYEOF

cat > /tmp/docker-compose-demo/app/requirements.txt << 'EOF'
flask==3.1.0
redis==5.2.1
EOF

# Dockerfile
cat > /tmp/docker-compose-demo/Dockerfile << 'DEOF'
FROM python:3.12-slim
WORKDIR /app
COPY app/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ .
EXPOSE 5000
CMD ["python", "main.py"]
DEOF

echo "Compose demo files created."

### 5.1 docker-compose.yml

In [ ]:
%%bash
cat > /tmp/docker-compose-demo/docker-compose.yml << 'EOF'
services:
  web:
    build: .
    ports:
      - "5050:5000"
    environment:
      - REDIS_HOST=redis
    depends_on:
      redis:
        condition: service_healthy
    restart: unless-stopped

  redis:
    image: redis:7-alpine
    volumes:
      - redis-data:/data
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 10s
      timeout: 3s
      retries: 3

volumes:
  redis-data:
EOF

cat /tmp/docker-compose-demo/docker-compose.yml

### 5.2 Compose Up & Test

In [ ]:
%%bash
cd /tmp/docker-compose-demo
docker compose up -d --build

echo "\nWaiting for services..."
sleep 5

# Hit the endpoint 3 times — watch the counter
for i in 1 2 3; do
    echo "Request $i:"
    curl -s http://localhost:5050/ | python3 -m json.tool
done

In [ ]:
%%bash
# Useful compose commands
cd /tmp/docker-compose-demo

echo "=== Service Status ==="
docker compose ps

echo "\n=== Logs (last 5 lines per service) ==="
docker compose logs --tail 5

In [ ]:
%%bash
# Teardown everything
cd /tmp/docker-compose-demo
docker compose down -v
echo "All services stopped and volumes removed."

---
## 6. Multi-Stage Builds

Reduces final image size by separating build and runtime stages.

In [ ]:
%%bash
mkdir -p /tmp/multistage-demo

# A simple Go app to demonstrate multi-stage
cat > /tmp/multistage-demo/main.go << 'GOEOF'
package main

import (
    "fmt"
    "net/http"
)

func main() {
    http.HandleFunc("/", func(w http.ResponseWriter, r *http.Request) {
        fmt.Fprintf(w, "Hello from a multi-stage build!")
    })
    fmt.Println("Listening on :8080")
    http.ListenAndServe(":8080", nil)
}
GOEOF

cat > /tmp/multistage-demo/Dockerfile << 'DEOF'
# ---- Stage 1: Build ----
FROM golang:1.22-alpine AS builder
WORKDIR /src
COPY main.go .
RUN CGO_ENABLED=0 go build -o /app main.go

# ---- Stage 2: Runtime ----
FROM scratch
COPY --from=builder /app /app
EXPOSE 8080
ENTRYPOINT ["/app"]
DEOF

echo "Multi-stage demo files created."
cat /tmp/multistage-demo/Dockerfile

In [ ]:
%%bash
# Build and compare sizes
docker build -t go-app:multistage /tmp/multistage-demo/

echo "\n=== Image Size ==="
docker images go-app:multistage
echo "\n(Compare: golang:1.22-alpine is ~250MB, this final image is ~7MB)"

In [ ]:
%%bash
# Test it
docker run -d --name go-app -p 8081:8080 go-app:multistage
sleep 1
curl -s http://localhost:8081/

# Cleanup
docker stop go-app && docker rm go-app

---
## 7. Security & Best Practices

### 7.1 Run as Non-Root User

In [ ]:
%%bash
cat << 'EOF'
# Best practice Dockerfile with non-root user:

FROM python:3.12-slim

# Create a non-root user
RUN groupadd -r appuser && useradd -r -g appuser appuser

WORKDIR /app
COPY --chown=appuser:appuser . .
RUN pip install --no-cache-dir -r requirements.txt

# Switch to non-root
USER appuser

CMD ["python", "main.py"]
EOF

### 7.2 Resource Limits

In [ ]:
%%bash
# Run with memory and CPU limits
docker run --rm --memory=128m --cpus=0.5 alpine:3.19 \
    sh -c 'echo "Running with 128MB RAM and 0.5 CPU cores"'

### 7.3 Scan for Vulnerabilities

In [ ]:
%%bash
# Docker Scout (built-in vulnerability scanner)
docker scout quickview alpine:3.19 2>/dev/null || echo "Docker Scout not available — install via Docker Desktop."

### 7.4 Best Practices Cheat Sheet

| Practice | Why |
|----------|-----|
| Use specific image tags (`python:3.12-slim`) | Reproducible builds |
| Copy `requirements.txt` before code | Layer caching — deps don't re-install on code change |
| Use `.dockerignore` | Smaller build context, no secrets leaked |
| Multi-stage builds | Smaller final images |
| Run as non-root `USER` | Limit blast radius of exploits |
| One process per container | Easier scaling and debugging |
| Use `HEALTHCHECK` | Orchestrators know when the app is ready |
| `--no-cache-dir` with pip | Smaller image layers |

---
## 8. Cleanup

In [ ]:
%%bash
# Remove demo images
docker rmi demo-flask:v1 go-app:multistage 2>/dev/null

# Prune unused resources
echo "\n=== Disk usage before prune ==="
docker system df

# Uncomment to reclaim space:
# docker system prune -af --volumes

# Remove temp files
rm -rf /tmp/docker-demo /tmp/docker-compose-demo /tmp/multistage-demo
echo "\nCleanup complete."

---
## Quick Reference

```bash
# Images
docker build -t name:tag .        # Build image
docker images                      # List images
docker rmi image                   # Remove image

# Containers
docker run -d -p 80:80 --name c img   # Run detached
docker exec -it c bash                 # Shell into container
docker logs -f c                       # Follow logs
docker stop c && docker rm c           # Stop & remove

# Volumes & Networks
docker volume create v                 # Create volume
docker network create n                # Create network

# Compose
docker compose up -d --build           # Build & start
docker compose down -v                 # Stop & remove volumes
docker compose logs -f                 # Follow all logs

# Housekeeping
docker system df                       # Disk usage
docker system prune -af                # Remove all unused
```